##1. 데이터 세트 로딩 및 키 값 확인

scikit-learn의 fetch_20newsgroups API를 사용해 텍스트 데이터를 로드함.

반환되는 Bunch 객체는 파이썬 딕셔너리와 유사한 구조를 가짐.

keys()를 통해 데이터 구성 요소(data, target, target_names 등)를 사전 파악함.

In [1]:
from sklearn.datasets import fetch_20newsgroups

news_data = fetch_20newsgroups(subset='all', random_state=156)

print(news_data.keys())

dict_keys(['data', 'filenames', 'target_names', 'target', 'DESCR'])


##2. Target 클래스 값/분포 및 첫 번째 데이터 확인

종속 변수(Target)의 범주와 데이터 분포를 확인하여 불균형 데이터 여부를 점검함.

텍스트 원본(data)을 출력해 뉴스그룹 기사의 구성(헤더, 푸터, 본문 등)을 형태적 측면에서 검토함.

In [2]:
import pandas as pd

print('target 클래스의 값과 분포도 \n', pd.Series(news_data.target).value_counts().sort_index())
print('target 클래스의 이름들 \n', news_data.target_names)

# 개별 데이터가 텍스트로 어떻게 구성돼 있는지 확인
print(news_data.data[0])

target 클래스의 값과 분포도 
 0     799
1     973
2     985
3     982
4     963
5     988
6     975
7     990
8     996
9     994
10    999
11    991
12    984
13    990
14    987
15    997
16    910
17    940
18    775
19    628
Name: count, dtype: int64
target 클래스의 이름들 
 ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']
From: egreen@east.sun.com (Ed Green - Pixel Cruncher)
Subject: Re: Observation re: helmets
Organization: Sun Microsystems, RTP, NC
Lines: 21
Distribution: world
Reply-To: egreen@east.sun.com
NNTP-Posting-Host: laser.east.sun.com

In article 211353@mavenry.altcit.eskimo.com, maven@mavenry.altcit.eskimo.com (Norman Hamer) writes:
> 
> The qu

##3. 학습/테스트 데이터 세트 분리 및 텍스트 전처리

순수 본문 텍스트만으로 모델을 학습시키기 위한 전처리 과정임.

헤더, 푸터, 인용구 등의 메타 정보는 정답을 쉽게 유추할 수 있는 힌트(Data Leakage)가 되므로 remove 파라미터로 제거함.

기계학습 모델의 범용성을 높이기 위한 필수 과정.

In [3]:
from sklearn.datasets import fetch_20newsgroups

# subset='train'으로 학습용 데이터만 추출, remove=('headers', 'footers', 'quotes')로 내용만 추출
train_news = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'), random_state=156)
X_train = train_news.data
y_train = train_news.target

# subset='test'으로 테스트 데이터만 추출, remove=('headers', 'footers', 'quotes')로 내용만 추출
test_news = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'), random_state=156)
X_test = test_news.data
y_test = test_news.target

print('학습 데이터 크기 {0}, 테스트 데이터 크기 {1}'.format(len(train_news.data), len(test_news.data)))

학습 데이터 크기 11314, 테스트 데이터 크기 7532


##4. CountVectorizer 피처 벡터화 및 로지스틱 회귀

CountVectorizer: 단어의 출현 빈도(Count)를 기준으로 텍스트를 수치 행렬(Bag of Words)로 변환함.

주의점: 테스트 데이터는 반드시 학습 데이터로 fit()된 객체를 사용해 transform()만 수행해야 피처 개수가 동일하게 유지됨.

텍스트와 같은 희소 행렬(Sparse Matrix) 분류에 뛰어난 성능을 보이는 로지스틱 회귀 모델을 기준 모델(Baseline)로 적용함.

In [4]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings

warnings.filterwarnings('ignore')

# Count Vectorization으로 피처 벡터화 변환 수행.
cnt_vect = CountVectorizer()
cnt_vect.fit(X_train)
X_train_cnt_vect = cnt_vect.transform(X_train)

# 학습 데이터로 fit()된 CountVectorizer를 이용해 테스트 데이터를 피처 벡터화 변환 수행.
X_test_cnt_vect = cnt_vect.transform(X_test)
print('학습 데이터 텍스트의 CountVectorizer Shape:', X_train_cnt_vect.shape)

# LogisticRegression을 이용하여 학습/예측/평가 수행.
lr_clf = LogisticRegression(solver='liblinear')
lr_clf.fit(X_train_cnt_vect, y_train)
pred = lr_clf.predict(X_test_cnt_vect)

print('CountVectorized Logistic Regression의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test, pred)))

학습 데이터 텍스트의 CountVectorizer Shape: (11314, 101631)
CountVectorized Logistic Regression의 예측 정확도는 0.617


##5. TfidfVectorizer 피처 벡터화 기본 파라미터 적용 모델

TF-IDF (Term Frequency-Inverse Document Frequency): 단순 빈도 기반의 한계를 보완하는 벡터화 방식.

개별 문서에서 자주 등장하는 단어에 높은 가중치를 주되, 모든 문서에서 범용적으로 자주 등장하는 단어(예: a, the)에는 페널티를 부여함.

일반적으로 텍스트 분류에서 Count 기반보다 예측 성능이 우수함.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF 벡터화를 적용해 학습 데이터 세트와 테스트 데이터 세트 변환.
tfidf_vect = TfidfVectorizer()
tfidf_vect.fit(X_train)
X_train_tfidf_vect = tfidf_vect.transform(X_train)
X_test_tfidf_vect = tfidf_vect.transform(X_test)

# LogisticRegression을 이용해 학습/예측/평가 수행.
lr_clf = LogisticRegression(solver='liblinear')
lr_clf.fit(X_train_tfidf_vect, y_train)
pred = lr_clf.predict(X_test_tfidf_vect)

print('TF-IDF Logistic Regression의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test, pred)))

TF-IDF Logistic Regression의 예측 정확도는 0.678


##6. TfidfVectorizer 파라미터 튜닝 적용 모델

텍스트 전처리 및 벡터화 성능을 높이기 위한 파라미터 튜닝 과정임.

stop_words='english': 무의미한 영어 불용어 제거.

ngram_range=(1,2): 단어 묶음(Unigram+Bigram)을 고려해 문맥을 반영함.

max_df=300: 전체 문서에 걸쳐 너무 높은 빈도로 나타나는 단어는 피처에서 제외함.

In [7]:
# stop words 필터링을 추가하고 ngram을 기본 (1, 1)에서 (1, 2)로 변경해 피처 벡터화 적용.
tfidf_vect = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_df=300)
tfidf_vect.fit(X_train)
X_train_tfidf_vect = tfidf_vect.transform(X_train)
X_test_tfidf_vect = tfidf_vect.transform(X_test)

lr_clf = LogisticRegression(solver='liblinear')
lr_clf.fit(X_train_tfidf_vect, y_train)
pred = lr_clf.predict(X_test_tfidf_vect)

print('TF-IDF Vectorized Logistic Regression 의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test, pred)))

TF-IDF Vectorized Logistic Regression 의 예측 정확도는 0.690


##7. GridSearchCV를 이용한 하이퍼 파라미터 최적화

GridSearchCV: 교차 검증을 통해 과적합을 방지하고 최적의 하이퍼 파라미터를 탐색함.  

로지스틱 회귀의 C 파라미터(규제 강도) 조절. C 값이 클수록 규제가 약해지고, 작을수록 규제가 강해짐.

In [8]:
from sklearn.model_selection import GridSearchCV

# 최적 C 값 도출 튜닝 수행. CV는 3 폴드 세트로 설정.
params = {'C': [0.01, 0.1, 1, 5, 10]}
grid_cv_lr = GridSearchCV(lr_clf, param_grid=params, cv=3, scoring='accuracy', verbose=1)
grid_cv_lr.fit(X_train_tfidf_vect, y_train)

print('Logistic Regression best C parameter:', grid_cv_lr.best_params_)

# 최적 C 값으로 학습된 grid_cv로 예측 및 정확도 평가.
pred = grid_cv_lr.predict(X_test_tfidf_vect)
print('TF-IDF Vectorized Logistic Regression의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test, pred)))

Fitting 3 folds for each of 5 candidates, totalling 15 fits
Logistic Regression best C parameter: {'C': 10}
TF-IDF Vectorized Logistic Regression의 예측 정확도는 0.704


##8. Pipeline을 이용한 간편한 전처리 및 ML 학습

피처 벡터화(TfidfVectorizer)와 ML 모델 학습(LogisticRegression)을 연결하는 Pipeline 구축.  

전처리와 모델링을 스트림 기반으로 묶어 코드 가독성을 높이고, 테스트 데이터에 잘못된 fit이 적용되는 실수를 구조적으로 방지함.

In [9]:
from sklearn.pipeline import Pipeline

# TfidfVectorizer 객체를 tfidf_vect로, LogisticRegression 객체를 lr_clf로 생성하는 Pipeline 생성
pipeline = Pipeline([
    ('tfidf_vect', TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_df=300)),
    ('lr_clf', LogisticRegression(solver='liblinear', C=10))
])

# 별도의 TfidfVectorizer 객체의 fit(), transform()과 LogisticRegression의 fit(), predict()가 필요 없음.
# pipeline의 fit()과 predict()만으로 한꺼번에 피처 벡터화와 ML 학습/예측이 가능.
pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)

print('Pipeline을 통한 Logistic Regression의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test, pred)))

Pipeline을 통한 Logistic Regression의 예측 정확도는 0.704


##9. Pipeline과 GridSearchCV 결합을 통한 통합 최적화

텍스트 벡터화 파라미터(N-gram, max_df 등)와 머신러닝 알고리즘의 파라미터(C)를 동시에 최적화하는 기법.

param_grid 설정 시 객체 이름과 파라미터명 사이에 언더바 두 개(__)를 연결하여 식별함.

최상의 성능 조합을 찾을 수 있으나, 탐색 공간이 기하급수적으로 커져 연산 시간이 크게 증가한다는 단점이 있음.

In [10]:
# Pipeline에 기술된 각각의 객체 변수에 언더바(__) 2개를 연달아 붙여 GridSearchCV에 사용될 파라미터 이름 설정.
pipeline = Pipeline([
    ('tfidf_vect', TfidfVectorizer(stop_words='english')),
    ('lr_clf', LogisticRegression(solver='liblinear'))
])

params = {
    'tfidf_vect__ngram_range': [(1, 1), (1, 2), (1, 3)],
    'tfidf_vect__max_df': [100, 300, 700],
    'lr_clf__C': [1, 5, 10]
}

# GridSearchCV의 생성자에 Estimator가 아닌 Pipeline 객체 입력
grid_cv_pipe = GridSearchCV(pipeline, param_grid=params, cv=3, scoring='accuracy', verbose=1)
grid_cv_pipe.fit(X_train, y_train)

print(grid_cv_pipe.best_params_, grid_cv_pipe.best_score_)

pred = grid_cv_pipe.predict(X_test)
print('Pipeline을 통한 Logistic Regression의 예측 정확도는 {0:.3f}'.format(accuracy_score(y_test, pred)))

Fitting 3 folds for each of 27 candidates, totalling 81 fits
{'lr_clf__C': 10, 'tfidf_vect__max_df': 700, 'tfidf_vect__ngram_range': (1, 2)} 0.7550828826229531
Pipeline을 통한 Logistic Regression의 예측 정확도는 0.702
